# Aggressive Driving Classifier
**Accuracy: 89.40%** (baseline: 85.62%)

Improvements over baseline:
- Larger window (40 → 80 samples) to capture full motion events
- Denser training stride (20 → 10) to double training samples
- Rich per-axis features: skew, kurtosis, RMS, quartiles, derivative energy
- Resultant magnitude + jerk (rate-of-change) features
- Cross-axis correlation features
- More trees (20 → 200)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import skew, kurtosis

In [ ]:
def get_window_features(df, window_size=80, step=20):
    """
    Extract 123 statistical features from a sliding window over IMU data.

    Features per window:
      - 15 stats × 6 axes        = 90  (mean, std, range, max, min, median,
                                         skew, kurtosis, RMS, Q1, Q3,
                                         total variation, MAD, deriv energy, mean abs diff)
      - 10 stats × 2 magnitudes  = 20  (acc_mag, gyro_mag)
      - 3 stats  × 2 jerk signals = 6  (mean, std, max-abs jerk)
      - 6 cross-axis correlations + 1 acc/gyro-mag correlation = 7
    """
    features, labels = [], []
    cols = ['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ']

    for i in range(0, len(df) - window_size, step):
        window = df.iloc[i : i + window_size]
        stats = []

        # ── Per-axis statistical features ────────────────────────────────
        for col in cols:
            data = window[col].values
            stats.extend([
                np.mean(data),
                np.std(data),
                np.max(data) - np.min(data),           # range
                np.max(data),
                np.min(data),
                np.median(data),
                skew(data),
                kurtosis(data),
                np.sqrt(np.mean(data**2)),             # RMS
                np.percentile(data, 25),               # Q1
                np.percentile(data, 75),               # Q3
                np.sum(np.abs(np.diff(data))),         # total variation
                np.mean(np.abs(data - np.mean(data))), # mean absolute deviation
                np.sum(np.diff(data)**2),              # derivative energy
                np.mean(np.abs(np.diff(data))),        # mean absolute difference
            ])

        # ── Resultant magnitude features ─────────────────────────────────
        acc  = window[['AccX', 'AccY', 'AccZ']].values
        gyro = window[['GyroX', 'GyroY', 'GyroZ']].values
        acc_mag  = np.sqrt((acc**2).sum(axis=1))
        gyro_mag = np.sqrt((gyro**2).sum(axis=1))

        def mag_feats(m):
            return [
                np.mean(m), np.std(m), np.max(m), np.min(m),
                np.max(m) - np.min(m),
                np.sqrt(np.mean(m**2)),
                skew(m), kurtosis(m),
                np.sum(np.abs(np.diff(m))),
                np.mean(np.abs(np.diff(m))),
            ]

        stats.extend(mag_feats(acc_mag))
        stats.extend(mag_feats(gyro_mag))

        # ── Jerk (rate of change of magnitude) ───────────────────────────
        jerk_acc  = np.diff(acc_mag)
        jerk_gyro = np.diff(gyro_mag)
        stats.extend([np.mean(jerk_acc),  np.std(jerk_acc),  np.max(np.abs(jerk_acc))])
        stats.extend([np.mean(jerk_gyro), np.std(jerk_gyro), np.max(np.abs(jerk_gyro))])

        # ── Cross-axis correlations ───────────────────────────────────────
        axis_pairs = [
            ('AccX','AccY'), ('AccX','AccZ'), ('AccY','AccZ'),
            ('GyroX','GyroY'), ('GyroX','GyroZ'), ('GyroY','GyroZ'),
        ]
        for a, b in axis_pairs:
            da, db = window[a].values, window[b].values
            corr = np.corrcoef(da, db)[0, 1] if (np.std(da) > 0 and np.std(db) > 0) else 0
            stats.append(corr)

        # acc_mag vs gyro_mag correlation
        corr_ag = (
            np.corrcoef(acc_mag, gyro_mag)[0, 1]
            if (np.std(acc_mag) > 0 and np.std(gyro_mag) > 0) else 0
        )
        stats.append(corr_ag)

        features.append(stats)
        # Label: 1 = Aggressive, 0 = Non-aggressive
        labels.append(1 if window['Class'].mode()[0] == 'AGGRESSIVE' else 0)

    return np.array(features), np.array(labels)

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
train_df = pd.read_csv('train_motion_data.csv')
test_df  = pd.read_csv('test_motion_data.csv')

# Dense stride for training (more samples), standard stride for test
X_train, y_train = get_window_features(train_df, window_size=80, step=10)
X_test,  y_test  = get_window_features(test_df,  window_size=80, step=20)

print(f"Features  : {X_train.shape[1]}")
print(f"Train windows : {len(X_train)}  |  aggressive: {y_train.sum()}")
print(f"Test  windows : {len(X_test)}   |  aggressive: {y_test.sum()}")

Features  : 123
Train windows : 357  |  aggressive: 112
Test  windows : 151   |  aggressive: 39


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=200,   # more trees → lower variance
    max_depth=10,       # still fits ESP32 memory (keep ≤ 12)
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)
print(f"Buzzer System Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print(classification_report(y_test, y_pred, target_names=['Non-Aggressive', 'Aggressive']))

Buzzer System Accuracy: 89.40%

                precision    recall  f1-score   support

Non-Aggressive       0.94      0.92      0.93       112
    Aggressive       0.78      0.82      0.80        39

      accuracy                           0.89       151
     macro avg       0.86      0.87      0.86       151
  weighted avg       0.90      0.89      0.89       151



## ESP32 Deployment Notes

| Parameter | Value |
|-----------|-------|
| Window size | 80 samples |
| Step / stride | 20 samples |
| Feature vector size | 123 floats |
| Trees | 200 |
| Max depth | 10 |

To deploy on the ESP32, export the trained model with `emlearn` or `micromlgen`:

```python
from micromlgen import port
print(port(model))
```

The feature extraction in `get_window_features()` must be reimplemented in C on the ESP32 — all operations used (mean, std, min, max, RMS, skew, kurtosis, correlation) are straightforward one-pass or two-pass calculations over the 80-sample buffer.